# Proyecto 2 – CC3045 Inteligencia Artificial
## Sistema de IA para Ciberseguridad de Red de Servidores Distribuidos
**Universidad del Valle de Guatemala**

Este notebook integra los tres módulos del Proyecto 2:

| Tarea | Tema | Algoritmos |
|-------|------|------------|
| Task 1 | Configuración Segura de Red (CSP) | Backtracking Básico · Backtracking con FC+MRV |
| Task 2 | Defensa Adversarial – Juegos de Suma Cero | Minimax Puro · Minimax con Poda Alfa-Beta |
| Task 3 | Incertidumbre y Latencia | Expectiminimax · MDPs · Ecuación de Bellman |

**Entorno**: Python 3.12  
**Dependencias externas**: `networkx`, `matplotlib`

---
# Tarea 1: Configuración Segura de la Red usando CSP y Factor Graphs

# Tarea 1: Configuracion Segura de la Red usando CSP y Factor Graphs

## Modelado Formal del CSP

**Variables**: X = {x₁, x₂, ..., xₙ} donde cada xᵢ representa un servidor en la red

**Dominios**: D(xᵢ) = {Rojo, Verde, Azul, Amarillo} para todo xᵢ ∈ X
(4 protocolos de seguridad)

**Restricciones**: C = {(xᵢ, xⱼ) : xᵢ ≠ xⱼ para toda arista (i,j) en el grafo}
Dos servidores directamente conectados NO pueden tener el mismo protocolo.

**Factor Graph**:
- Nodos variable: uno por servidor
- Nodos factor: uno por restriccion de adyacencia (una funcion indicadora fᵢⱼ(xᵢ, xⱼ))
- Factor fᵢⱼ(xᵢ, xⱼ) = 1 si xᵢ ≠ xⱼ, 0 en caso contrario
- La asignacion es valida si ∏ fᵢⱼ(xᵢ, xⱼ) = 1 para todos los factores

## Algoritmos Implementados
1. **Backtracking Basico**: busqueda exhaustiva sin optimizaciones
2. **Backtracking Optimizado**: con Forward Checking (lookahead) + MRV heuristic

In [1]:
import random
import time
import copy
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

## Seccion 1: Generacion del Grafo de la Red

In [2]:
def _es_4_colorable_rapido(G):
    """
    Verifica si el grafo es 4-colorable via greedy coloring.
    """
    coloreo = nx.coloring.greedy_color(G, strategy="largest_first")
    return max(coloreo.values()) + 1 <= 4

def generar_grafo_red(num_nodos_min=15, num_nodos_max=20, semilla=42):
    """
    Genera un grafo aleatorio conexo (Erdos-Renyi, p=0.25) con 15-20 nodos.
    """
    random.seed(semilla)

    # Elige numero de nodos en el rango dado
    num_nodos = random.randint(num_nodos_min, num_nodos_max)

    # Erdos-Renyi con p=0.25, busca grafo conexo y 4-colorable
    while True:
        G = nx.erdos_renyi_graph(num_nodos, p=0.25, seed=semilla)
        
        if nx.is_connected(G) and _es_4_colorable_rapido(G):
            break
        semilla += 1  

    print(f"Grafo generado: {num_nodos} nodos, {G.number_of_edges()} aristas")
    print(f"Grado promedio: {2 * G.number_of_edges() / num_nodos:.2f}")
    return G

def describir_grafo(G):
    """Imprime estadisticas del grafo."""
    print("Descripcion del Grafo de la Red")
    print(f"  Numero de servidores (nodos): {G.number_of_nodes()}")
    print(f"  Numero de conexiones (aristas): {G.number_of_edges()}")
    print(f"  Grado minimo: {min(dict(G.degree()).values())}")
    print(f"  Grado maximo: {max(dict(G.degree()).values())}")
    print(f"  Grado promedio: {sum(dict(G.degree()).values()) / G.number_of_nodes():.2f}")
    print(f"  ¿Es conexo? {nx.is_connected(G)}")
    print(f"  Coeficiente de clustering promedio: {nx.average_clustering(G):.4f}")

## Seccion 2: Modelado del CSP

In [3]:
# Protocolos de seguridad
PROTOCOLOS = ["Rojo", "Verde", "Azul", "Amarillo"]
COLOR_MAP = {
    "Rojo": "#e74c3c",
    "Verde": "#2ecc71",
    "Azul": "#3498db",
    "Amarillo": "#f1c40f"
}

class CSPRedSegura:
    """
    Modela coloracion de grafos como CSP.
    Variables X = nodos, Dominios D = {Rojo, Verde, Azul, Amarillo},
    Restricciones: xi != xj para cada arista (i,j).
    """

    def __init__(self, grafo):
        self.grafo = grafo
        
        self.variables = list(grafo.nodes())
        
        self.dominios = {v: list(PROTOCOLOS) for v in self.variables}
        
        self.vecinos = {v: list(grafo.neighbors(v)) for v in self.variables}
        
        self.restricciones = list(grafo.edges())

    def es_consistente(self, variable, valor, asignacion):
        """
        Retorna True si asignar valor a variable no viola restricciones con vecinos asignados.
        """
        for vecino in self.vecinos[variable]:
            if vecino in asignacion and asignacion[vecino] == valor:
                return False  # Factor = 0, restriccion violada
        return True  # Todos los factores = 1

    def es_solucion_completa(self, asignacion):
        """Retorna True si la asignacion es completa y valida."""
        if len(asignacion) != len(self.variables):
            return False
        # Verificar todas las restricciones (producto de todos los factores = 1)
        for u, v in self.restricciones:
            if u in asignacion and v in asignacion:
                if asignacion[u] == asignacion[v]:
                    return False
        return True

    def obtener_factores(self):
        """
        Retorna la lista de factores del Factor Graph.
        """
        factores = []
        for u, v in self.restricciones:
            factores.append({
                "nombre": f"f({u},{v})",
                "variables": (u, v),
                "funcion": lambda a, b: 1 if a != b else 0
            })
        return factores

## Seccion 3: Backtracking Basico (sin optimizaciones)

In [4]:
class BacktrackingBasico:
    """
    Backtracking puro sin optimizaciones. Complejidad O(d^n).
    """

    def __init__(self, csp):
        self.csp = csp
        
        self.num_asignaciones = 0
        self.num_backtracks = 0
        self.tiempo_inicio = 0
        self.tiempo_fin = 0

    def reiniciar_metricas(self):
        self.num_asignaciones = 0
        self.num_backtracks = 0

    def seleccionar_variable(self, asignacion):
        """
        Selecciona la primera variable no asignada (orden fijo, sin MRV).
        """
        for v in self.csp.variables:
            if v not in asignacion:
                return v
        return None

    def backtrack(self, asignacion):
        """
        Backtracking recursivo: busca asignacion que satisfaga todos los factores.
        """
        # Caso base
        if len(asignacion) == len(self.csp.variables):
            return asignacion

        variable = self.seleccionar_variable(asignacion)

        for valor in self.csp.dominios[variable]:
            self.num_asignaciones += 1

            if self.csp.es_consistente(variable, valor, asignacion):
                
                asignacion[variable] = valor

                resultado = self.backtrack(asignacion)
                if resultado is not None:
                    return resultado

                # Backtrack
                del asignacion[variable]
                self.num_backtracks += 1

        return None  

    def resolver(self):
        """Resuelve el CSP con backtracking basico."""
        self.reiniciar_metricas()
        self.tiempo_inicio = time.perf_counter()
        solucion = self.backtrack({})
        self.tiempo_fin = time.perf_counter()
        return solucion

    def obtener_metricas(self):
        return {
            "algoritmo": "Backtracking Basico",
            "asignaciones": self.num_asignaciones,
            "backtracks": self.num_backtracks,
            "tiempo_s": self.tiempo_fin - self.tiempo_inicio
        }

## Seccion 4: Backtracking Optimizado (Forward Checking + MRV)

In [5]:
class BacktrackingOptimizado:
    """
    Backtracking con Forward Checking (lookahead) y MRV (fail-first).
    """

    def __init__(self, csp):
        self.csp = csp
        self.num_asignaciones = 0
        self.num_backtracks = 0
        self.tiempo_inicio = 0
        self.tiempo_fin = 0

    def reiniciar_metricas(self):
        self.num_asignaciones = 0
        self.num_backtracks = 0

    def seleccionar_variable_mrv(self, asignacion, dominios_actuales):
        """
        MRV: selecciona variable con dominio mas pequeño (fail-first).
        """
        variables_no_asignadas = [v for v in self.csp.variables if v not in asignacion]
        if not variables_no_asignadas:
            return None

        # Seleccionar variable con dominio mas pequeño (MRV)
        return min(variables_no_asignadas, key=lambda v: len(dominios_actuales[v]))

    def forward_checking(self, variable, valor, asignacion, dominios_actuales):
        """
        Forward checking: elimina valor del dominio de vecinos no asignados.
        Retorna (exito, dominios_podados). Si algun dominio queda vacio, retorna False.
        """
        dominios_podados = {}  # Para poder restaurar al hacer backtrack

        for vecino in self.csp.vecinos[variable]:
            if vecino not in asignacion:
                if valor in dominios_actuales[vecino]:
                    
                    if vecino not in dominios_podados:
                        dominios_podados[vecino] = []
                    dominios_podados[vecino].append(valor)
                    dominios_actuales[vecino].remove(valor)

                    # Dominio vacio: falla
                    if len(dominios_actuales[vecino]) == 0:
                        
                        self._restaurar_dominios(dominios_podados, dominios_actuales)
                        return False, {}

        return True, dominios_podados

    def _restaurar_dominios(self, dominios_podados, dominios_actuales):
        """Restaura dominios podados durante backtrack."""
        for vecino, valores in dominios_podados.items():
            for valor in valores:
                if valor not in dominios_actuales[vecino]:
                    dominios_actuales[vecino].append(valor)

    def backtrack(self, asignacion, dominios_actuales):
        """
        Backtracking recursivo con forward checking y MRV.
        """
        # Caso base
        if len(asignacion) == len(self.csp.variables):
            return asignacion

        # MRV
        variable = self.seleccionar_variable_mrv(asignacion, dominios_actuales)

        for valor in list(dominios_actuales[variable]):
            self.num_asignaciones += 1

            if self.csp.es_consistente(variable, valor, asignacion):
                
                asignacion[variable] = valor

                # Forward checking
                exito, dominios_podados = self.forward_checking(
                    variable, valor, asignacion, dominios_actuales
                )

                if exito:
                    resultado = self.backtrack(asignacion, dominios_actuales)
                    if resultado is not None:
                        return resultado

                # Backtrack
                self._restaurar_dominios(dominios_podados, dominios_actuales)
                del asignacion[variable]
                self.num_backtracks += 1

        return None  # Fallo

    def resolver(self):
        """Resuelve el CSP con backtracking optimizado (FC + MRV)."""
        self.reiniciar_metricas()
        
        dominios_actuales = {v: list(vals) for v, vals in self.csp.dominios.items()}
        self.tiempo_inicio = time.perf_counter()
        solucion = self.backtrack({}, dominios_actuales)
        self.tiempo_fin = time.perf_counter()
        return solucion

    def obtener_metricas(self):
        return {
            "algoritmo": "Backtracking Optimizado (FC + MRV)",
            "asignaciones": self.num_asignaciones,
            "backtracks": self.num_backtracks,
            "tiempo_s": self.tiempo_fin - self.tiempo_inicio
        }

## Seccion 5: Factor Graph — Representacion Formal

In [6]:
class FactorGraph:
    """
    Factor Graph bipartito: nodos variable (servidores) y nodos factor (restricciones).
    f_ij(a,b) = 1 si a != b, 0 en caso contrario.
    """

    def __init__(self, csp):
        self.csp = csp
        self.nodos_variable = csp.variables
        self.nodos_factor = list(csp.grafo.edges())
        
        self.conexiones = self._construir_conexiones()

    def _construir_conexiones(self):
        """
        Construye conexiones del Factor Graph.
        """
        conexiones = defaultdict(list)
        for u, v in self.nodos_factor:
            factor_nombre = f"f({u},{v})"
            conexiones[factor_nombre].extend([u, v])
        return dict(conexiones)

    def evaluar(self, asignacion):
        """
        Evalua el producto de todos los factores. Retorna 1 si es valida, 0 si no.
        """
        for u, v in self.nodos_factor:
            if u in asignacion and v in asignacion:
                if asignacion[u] == asignacion[v]:
                    return 0  # Factor = 0, restriccion violada
        return 1

    def describir(self):
        """Imprime la estructura del Factor Graph."""
        print("Factor Graph")
        print(f"  Nodos variable (servidores): {len(self.nodos_variable)}")
        print(f"  Nodos factor (restricciones): {len(self.nodos_factor)}")
        print(f"  Total aristas del Factor Graph: {2 * len(self.nodos_factor)}")
        print("\n  Factores fᵢⱼ(xᵢ, xⱼ) = 1 si xᵢ ≠ xⱼ, 0 en caso contrario:")
        for i, (u, v) in enumerate(self.nodos_factor[:5]):
            print(f"    f({u},{v}) conecta variable x{u} con variable x{v}")
        if len(self.nodos_factor) > 5:
            print(f"    ... y {len(self.nodos_factor) - 5} factores mas")

    def visualizar(self, ax=None):
        """Dibuja el Factor Graph como grafo bipartito."""
        FG = nx.Graph()

        for v in self.nodos_variable:
            FG.add_node(f"x{v}", tipo="variable")

        for u, v in self.nodos_factor:
            fname = f"f({u},{v})"
            FG.add_node(fname, tipo="factor")
            FG.add_edge(f"x{u}", fname)
            FG.add_edge(f"x{v}", fname)

        if ax is None:
            fig, ax = plt.subplots(1, 1, figsize=(12, 8))

        var_nodes = [f"x{v}" for v in self.nodos_variable]
        fac_nodes = [f"f({u},{v})" for u, v in self.nodos_factor]

        pos = {}
        for i, n in enumerate(var_nodes):
            pos[n] = (0, i - len(var_nodes) / 2)
        for i, n in enumerate(fac_nodes):
            pos[n] = (2, i - len(fac_nodes) / 2)

        nx.draw_networkx_nodes(FG, pos, nodelist=var_nodes,
                               node_color="#3498db", node_size=300,
                               node_shape='o', ax=ax)
        nx.draw_networkx_nodes(FG, pos, nodelist=fac_nodes,
                               node_color="#e74c3c", node_size=200,
                               node_shape='s', ax=ax)
        nx.draw_networkx_edges(FG, pos, alpha=0.5, ax=ax)
        nx.draw_networkx_labels(FG, pos, font_size=6, ax=ax)

        ax.set_title("Factor Graph (bipartito)\nCirculos=Variables, Cuadros=Factores", fontsize=10)
        ax.axis("off")

## Seccion 6: Analisis y Comparacion de Rendimiento

In [7]:
def comparar_algoritmos(csp, verbose=True):
    """
    Ejecuta ambos algoritmos y compara metricas de rendimiento.
    """
    print("Comparacion de Algoritmos")
    print("Resolviendo CSP con Backtracking Basico...")

    bt_basico = BacktrackingBasico(csp)
    solucion_basica = bt_basico.resolver()
    metricas_basico = bt_basico.obtener_metricas()

    print("Resolviendo CSP con Backtracking Optimizado (FC + MRV)...")

    bt_opt = BacktrackingOptimizado(csp)
    solucion_opt = bt_opt.resolver()
    metricas_opt = bt_opt.obtener_metricas()

    valida_basica = csp.es_solucion_completa(solucion_basica) if solucion_basica else False
    valida_opt = csp.es_solucion_completa(solucion_opt) if solucion_opt else False

    if verbose:
        print(f"{'Metrica':<35} {'Basico':>15} {'Optimizado (FC+MRV)':>17}")
        print(f"{'Solucion encontrada':<35} {'Si' if solucion_basica else 'No':>15} {'Si' if solucion_opt else 'No':>17}")
        print(f"{'Solucion valida':<35} {'Si' if valida_basica else 'No':>15} {'Si' if valida_opt else 'No':>17}")
        print(f"{'Asignaciones intentadas':<35} {metricas_basico['asignaciones']:>15,} {metricas_opt['asignaciones']:>17,}")
        print(f"{'Numero de backtracks':<35} {metricas_basico['backtracks']:>15,} {metricas_opt['backtracks']:>17,}")
        print(f"{'Tiempo de ejecucion (s)':<35} {metricas_basico['tiempo_s']:>15.6f} {metricas_opt['tiempo_s']:>17.6f}")

        if metricas_opt['tiempo_s'] > 0 and metricas_basico['tiempo_s'] > 0:
            speedup = metricas_basico['tiempo_s'] / metricas_opt['tiempo_s']
            reduccion_asig = (1 - metricas_opt['asignaciones'] / max(metricas_basico['asignaciones'], 1)) * 100
            print(f"{'Speedup (basico / optimizado)':<35} {speedup:>15.2f}x")
            print(f"{'Reduccion en asignaciones':<35} {reduccion_asig:>14.1f}%")
        

    return {
        "solucion_basica": solucion_basica,
        "solucion_opt": solucion_opt,
        "metricas_basico": metricas_basico,
        "metricas_opt": metricas_opt
    }

def analisis_detallado(resultados, csp):
    """Imprime analisis detallado del rendimiento."""
    m_b = resultados["metricas_basico"]
    m_o = resultados["metricas_opt"]

    print("Analisis de Rendimiento")

    print("\n  Backtracking Basico:")
    print(f"    - Sin heuristicas: variables elegidas en orden fijo")
    print(f"    - Sin lookahead: inconsistencias detectadas tarde")
    print(f"    - Explora mas del arbol de busqueda de forma innecesaria")

    print("\n  Backtracking Optimizado (FC + MRV):")
    print(f"    - MRV: elige la variable mas restringida primero")
    print(f"      → Detecta fallos antes, reduce ramificacion")
    print(f"    - Forward Checking: elimina valores del dominio de vecinos")
    print(f"      → Poda subarboles imposibles sin explorarlos")

    if m_b["asignaciones"] > 0:
        ratio = m_o["asignaciones"] / m_b["asignaciones"]
        print(f"\n  El algoritmo optimizado exploro solo el {ratio*100:.1f}% de las")
        print(f"  asignaciones del basico ({m_o['asignaciones']:,} vs {m_b['asignaciones']:,})")

    if m_b["tiempo_s"] > 0:
        speedup = m_b["tiempo_s"] / m_o["tiempo_s"]
        print(f"  Aceleracion temporal: {speedup:.2f}x mas rapido")

    print("\n  Relacion con Factor Graphs:")
    print(f"    Forward Checking implementa propagacion local de mensajes")
    print(f"    similar al algoritmo Belief Propagation en Factor Graphs:")
    print(f"    μ_xᵢ→fᵢⱼ(v) = 0 si v fue eliminado del dominio D(xᵢ)")

## Seccion 7: Visualizaciones

In [8]:
def visualizar_grafo_coloreado(G, asignacion, titulo="Red con Protocolos de Seguridad", ax=None):
    """
    Dibuja el grafo con colores segun protocolo asignado.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 8))

    colores_nodos = []
    for nodo in G.nodes():
        protocolo = asignacion.get(nodo, "Sin asignar")
        colores_nodos.append(COLOR_MAP.get(protocolo, "#95a5a6"))

    pos = nx.spring_layout(G, seed=42, k=1.5)

    nx.draw_networkx_edges(G, pos, alpha=0.4, edge_color="gray", ax=ax)

    nx.draw_networkx_nodes(G, pos, node_color=colores_nodos,
                           node_size=500, ax=ax)

    labels = {n: f"S{n}" for n in G.nodes()}
    nx.draw_networkx_labels(G, pos, labels, font_size=8, font_color="white",
                            font_weight="bold", ax=ax)

    parches = [mpatches.Patch(color=COLOR_MAP[p], label=f"Protocolo {p}")
               for p in PROTOCOLOS]
    ax.legend(handles=parches, loc="upper left", fontsize=9)
    ax.set_title(titulo, fontsize=12, fontweight="bold")
    ax.axis("off")

def visualizar_metricas_comparacion(resultados):
    """
    Grafico de barras comparando metricas.
    """
    m_b = resultados["metricas_basico"]
    m_o = resultados["metricas_opt"]

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    fig.suptitle("Comparacion de Rendimiento: Backtracking Basico vs Optimizado",
                 fontsize=13, fontweight="bold")

    algoritmos = ["Basico", "Optimizado\n(FC + MRV)"]
    colores_barras = ["#e74c3c", "#2ecc71"]

    # Grafico 1: Asignaciones
    vals_asig = [m_b["asignaciones"], m_o["asignaciones"]]
    bars = axes[0].bar(algoritmos, vals_asig, color=colores_barras, edgecolor="black", width=0.5)
    axes[0].set_title("Asignaciones Intentadas", fontweight="bold")
    axes[0].set_ylabel("Numero de asignaciones")
    for bar, val in zip(bars, vals_asig):
        axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
                     f"{val:,}", ha="center", va="bottom", fontsize=9)

    # Grafico 2: Backtracks
    vals_bt = [m_b["backtracks"], m_o["backtracks"]]
    bars = axes[1].bar(algoritmos, vals_bt, color=colores_barras, edgecolor="black", width=0.5)
    axes[1].set_title("Numero de Backtracks", fontweight="bold")
    axes[1].set_ylabel("Numero de backtracks")
    for bar, val in zip(bars, vals_bt):
        axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
                     f"{val:,}", ha="center", va="bottom", fontsize=9)

    # Grafico 3: Tiempo
    vals_t = [m_b["tiempo_s"] * 1000, m_o["tiempo_s"] * 1000]  # en ms
    bars = axes[2].bar(algoritmos, vals_t, color=colores_barras, edgecolor="black", width=0.5)
    axes[2].set_title("Tiempo de Ejecucion", fontweight="bold")
    axes[2].set_ylabel("Tiempo (ms)")
    for bar, val in zip(bars, vals_t):
        axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
                     f"{val:.3f}", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.show()
    return fig

def visualizar_todo(G, resultados, factor_graph):
    """
    Genera figura compuesta con todas las visualizaciones.
    """
    fig = plt.figure(figsize=(18, 14))

    ax1 = fig.add_subplot(2, 3, 1)  # Grafo original
    ax2 = fig.add_subplot(2, 3, 2)  # Grafo con solucion basica
    ax3 = fig.add_subplot(2, 3, 3)  # Grafo con solucion optimizada
    ax4 = fig.add_subplot(2, 3, 4)  # Factor graph (parcial)
    ax5 = fig.add_subplot(2, 3, 5)  # Barras asignaciones
    ax6 = fig.add_subplot(2, 3, 6)  # Barras tiempo

    pos = nx.spring_layout(G, seed=42, k=1.5)
    nx.draw_networkx(G, pos, ax=ax1, node_color="#95a5a6", node_size=400,
                     font_size=7, font_color="white", font_weight="bold",
                     edge_color="gray", alpha=0.8,
                     labels={n: f"S{n}" for n in G.nodes()})
    ax1.set_title("Red de Servidores\n(sin protocolo asignado)", fontsize=10)
    ax1.axis("off")

    if resultados["solucion_basica"]:
        visualizar_grafo_coloreado(G, resultados["solucion_basica"],
                                   "Solucion: Backtracking Basico", ax=ax2)
    else:
        ax2.text(0.5, 0.5, "Sin solucion encontrada\n(grafo no 4-colorable)",
                 ha="center", va="center", transform=ax2.transAxes, fontsize=11)
        ax2.set_title("Solucion: Backtracking Basico", fontsize=10)
        ax2.axis("off")

    if resultados["solucion_opt"]:
        visualizar_grafo_coloreado(G, resultados["solucion_opt"],
                                   "Solucion: Backtracking Optimizado\n(FC + MRV)", ax=ax3)
    else:
        ax3.text(0.5, 0.5, "Sin solucion encontrada\n(grafo no 4-colorable)",
                 ha="center", va="center", transform=ax3.transAxes, fontsize=11)
        ax3.set_title("Solucion: Backtracking Optimizado\n(FC + MRV)", fontsize=10)
        ax3.axis("off")

    _visualizar_factor_graph_parcial(factor_graph, ax4)

    m_b = resultados["metricas_basico"]
    m_o = resultados["metricas_opt"]
    categorias = ["Asignaciones", "Backtracks"]
    vals_b = [m_b["asignaciones"], m_b["backtracks"]]
    vals_o = [m_o["asignaciones"], m_o["backtracks"]]
    x = range(len(categorias))
    width = 0.35
    bars1 = ax5.bar([xi - width/2 for xi in x], vals_b, width,
                    label="Basico", color="#e74c3c", edgecolor="black")
    bars2 = ax5.bar([xi + width/2 for xi in x], vals_o, width,
                    label="Optimizado", color="#2ecc71", edgecolor="black")
    ax5.set_xticks(list(x))
    ax5.set_xticklabels(categorias)
    ax5.set_title("Asignaciones y Backtracks", fontsize=10, fontweight="bold")
    ax5.legend(fontsize=8)
    for bar in list(bars1) + list(bars2):
        h = bar.get_height()
        ax5.text(bar.get_x() + bar.get_width() / 2, h * 1.01,
                 f"{h:,}", ha="center", va="bottom", fontsize=7)

    tiempos = [m_b["tiempo_s"] * 1000, m_o["tiempo_s"] * 1000]
    colores = ["#e74c3c", "#2ecc71"]
    algs = ["Basico", "Optimizado\n(FC+MRV)"]
    bars = ax6.bar(algs, tiempos, color=colores, edgecolor="black", width=0.5)
    ax6.set_title("Tiempo de Ejecucion (ms)", fontsize=10, fontweight="bold")
    ax6.set_ylabel("ms")
    for bar, val in zip(bars, tiempos):
        ax6.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
                 f"{val:.3f}ms", ha="center", va="bottom", fontsize=8)

    fig.suptitle("Tarea 1: CSP — Configuracion Segura de Red de Servidores",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()
    return fig

def _visualizar_factor_graph_parcial(factor_graph, ax):
    """Dibuja un subconjunto del Factor Graph."""
    
    vars_subset = list(factor_graph.nodos_variable)[:6]
    vars_set = set(vars_subset)

    FG = nx.Graph()

    for v in vars_subset:
        FG.add_node(f"x{v}", tipo="variable")

    factores_subset = []
    for u, v in factor_graph.nodos_factor:
        if u in vars_set and v in vars_set:
            fname = f"f({u},{v})"
            FG.add_node(fname, tipo="factor")
            FG.add_edge(f"x{u}", fname)
            FG.add_edge(f"x{v}", fname)
            factores_subset.append(fname)

    if len(FG.nodes()) < 2:
        ax.text(0.5, 0.5, "Factor Graph\n(sin factores en subconjunto)",
                ha="center", va="center", transform=ax.transAxes)
        ax.axis("off")
        return

    pos = nx.spring_layout(FG, seed=10)
    var_nodes = [f"x{v}" for v in vars_subset if f"x{v}" in FG]
    fac_nodes = [n for n in FG.nodes() if n not in var_nodes]

    nx.draw_networkx_nodes(FG, pos, nodelist=var_nodes, node_color="#3498db",
                           node_size=400, ax=ax)
    nx.draw_networkx_nodes(FG, pos, nodelist=fac_nodes, node_color="#e74c3c",
                           node_size=200, node_shape="s", ax=ax)
    nx.draw_networkx_edges(FG, pos, alpha=0.6, ax=ax)
    nx.draw_networkx_labels(FG, pos, font_size=6, font_color="white",
                            font_weight="bold", ax=ax)
    ax.set_title("Factor Graph (subconjunto)\n● Variables  ■ Factores", fontsize=9)
    ax.axis("off")

## Seccion 8: Benchmark con Grafos de Dificultad Variable

In [9]:
def benchmark_dificultad_variable():
    """
    Compara algoritmos en grafos de distinta densidad.
    """
    print("Benchmark: Dificultad Variable")
    print("(Comparacion en multiples instancias para ilustrar escalabilidad)")

    configuraciones = [
        {"p": 0.15, "semilla": 100, "label": "Baja densidad (p=0.15)"},
        {"p": 0.22, "semilla": 200, "label": "Media densidad (p=0.22)"},
        {"p": 0.28, "semilla": 300, "label": "Alta densidad (p=0.28)"},
    ]

    resultados_benchmark = []

    for cfg in configuraciones:
        
        semilla = cfg["semilla"]
        for intento in range(50):
            G_test = nx.erdos_renyi_graph(18, p=cfg["p"], seed=semilla + intento)
            if nx.is_connected(G_test) and _es_4_colorable_rapido(G_test):
                break

        csp_test = CSPRedSegura(G_test)

        bt_b = BacktrackingBasico(csp_test)
        bt_b.resolver()
        m_b = bt_b.obtener_metricas()

        bt_o = BacktrackingOptimizado(csp_test)
        bt_o.resolver()
        m_o = bt_o.obtener_metricas()

        speedup = m_b["tiempo_s"] / m_o["tiempo_s"] if m_o["tiempo_s"] > 0 else float("inf")
        reduccion = (1 - m_o["asignaciones"] / max(m_b["asignaciones"], 1)) * 100

        resultados_benchmark.append({
            "label": cfg["label"],
            "aristas": G_test.number_of_edges(),
            "asig_basico": m_b["asignaciones"],
            "asig_opt": m_o["asignaciones"],
            "bt_basico": m_b["backtracks"],
            "bt_opt": m_o["backtracks"],
            "speedup": speedup,
            "reduccion": reduccion
        })

    print(f"\n{'Instancia':<28} {'Aristas':>7} {'Asig.Bas':>9} {'Asig.Opt':>9} "
          f"{'BT.Bas':>7} {'BT.Opt':>7} {'Speedup':>8} {'Reduc.%':>8}")
    
    for r in resultados_benchmark:
        print(f"  {r['label']:<26} {r['aristas']:>7} {r['asig_basico']:>9,} {r['asig_opt']:>9,} "
              f"{r['bt_basico']:>7,} {r['bt_opt']:>7,} {r['speedup']:>7.2f}x {r['reduccion']:>7.1f}%")
    

    return resultados_benchmark

## Seccion 9: Funcion Principal

In [10]:
def main():
    """Ejecuta el pipeline completo: genera grafo, resuelve CSP, compara y visualiza."""
    print("Task 1: Configuracion segura de red (CSP + Factor Graphs)\n")

    G = generar_grafo_red(num_nodos_min=15, num_nodos_max=20, semilla=42)
    describir_grafo(G)

    csp = CSPRedSegura(G)
    factor_graph = FactorGraph(csp)

    print(f"\nCSP: {len(csp.variables)} variables, dominio={PROTOCOLOS}, {len(csp.restricciones)} restricciones")
    factor_graph.describir()

    print("\nResolviendo...")
    resultados = comparar_algoritmos(csp, verbose=True)

    analisis_detallado(resultados, csp)

    if resultados["solucion_basica"]:
        val_fg = factor_graph.evaluar(resultados["solucion_basica"])
        print(f"\nFactor Graph basica: prod(fij) = {val_fg} ({'valida' if val_fg == 1 else 'invalida'})")
    if resultados["solucion_opt"]:
        val_fg = factor_graph.evaluar(resultados["solucion_opt"])
        print(f"Factor Graph optimizada: prod(fij) = {val_fg} ({'valida' if val_fg == 1 else 'invalida'})")

    if resultados["solucion_opt"]:
        print("\nProtocolos asignados:")
        sol = resultados["solucion_opt"]
        for protocolo in PROTOCOLOS:
            servidores = [f"S{v}" for v, p in sorted(sol.items()) if p == protocolo]
            print(f"  {protocolo:8s}: {', '.join(servidores) if servidores else '(ninguno)'}")

    benchmark_dificultad_variable()

    try:
        visualizar_todo(G, resultados, factor_graph)
        plt.show()
    except Exception as e:
        print(f"Visualizacion omitida: {e}")

    return resultados

In [11]:
main()

Task 1: Configuracion segura de red (CSP + Factor Graphs)

Grafo generado: 20 nodos, 56 aristas
Grado promedio: 5.60
Descripcion del Grafo de la Red
  Numero de servidores (nodos): 20
  Numero de conexiones (aristas): 56
  Grado minimo: 2
  Grado maximo: 11
  Grado promedio: 5.60
  ¿Es conexo? True
  Coeficiente de clustering promedio: 0.2858

CSP: 20 variables, dominio=['Rojo', 'Verde', 'Azul', 'Amarillo'], 56 restricciones
Factor Graph
  Nodos variable (servidores): 20
  Nodos factor (restricciones): 56
  Total aristas del Factor Graph: 112

  Factores fᵢⱼ(xᵢ, xⱼ) = 1 si xᵢ ≠ xⱼ, 0 en caso contrario:
    f(0,2) conecta variable x0 con variable x2
    f(0,4) conecta variable x0 con variable x4
    f(0,8) conecta variable x0 con variable x8
    f(0,10) conecta variable x0 con variable x10
    f(0,11) conecta variable x0 con variable x11
    ... y 51 factores mas

Resolviendo...
Comparacion de Algoritmos
Resolviendo CSP con Backtracking Basico...
Resolviendo CSP con Backtracking Optimiz

/tmp/ipykernel_17594/526608530.py:149: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_17594/1362345256.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


{'solucion_basica': {0: 'Rojo',
  1: 'Rojo',
  2: 'Verde',
  3: 'Rojo',
  4: 'Verde',
  5: 'Rojo',
  6: 'Verde',
  7: 'Rojo',
  8: 'Azul',
  9: 'Verde',
  10: 'Azul',
  11: 'Amarillo',
  12: 'Rojo',
  13: 'Verde',
  14: 'Amarillo',
  15: 'Verde',
  16: 'Verde',
  17: 'Azul',
  18: 'Rojo',
  19: 'Amarillo'},
 'solucion_opt': {0: 'Rojo',
  2: 'Verde',
  8: 'Azul',
  11: 'Amarillo',
  10: 'Azul',
  17: 'Azul',
  12: 'Rojo',
  16: 'Verde',
  14: 'Amarillo',
  1: 'Rojo',
  6: 'Verde',
  5: 'Rojo',
  9: 'Verde',
  3: 'Rojo',
  4: 'Verde',
  13: 'Verde',
  15: 'Verde',
  19: 'Amarillo',
  18: 'Rojo',
  7: 'Rojo'},
 'metricas_basico': {'algoritmo': 'Backtracking Basico',
  'asignaciones': 42,
  'backtracks': 0,
  'tiempo_s': 2.622800002427539e-05},
 'metricas_opt': {'algoritmo': 'Backtracking Optimizado (FC + MRV)',
  'asignaciones': 20,
  'backtracks': 0,
  'tiempo_s': 9.066600000551261e-05}}

---
# Tarea 2: Defensa Adversarial – Juegos de Suma Cero (Minimax)

# Task 2 - Defensa Adversarial (Juegos de Suma Cero)
## CC3045 - Inteligencia Artificial, UVG

Este modulo implementa un juego de suma cero sobre una red de ciberseguridad.
- **MAX (Defensa)**: intenta maximizar la suma de valores de nodos capturados.
- **MIN (Atacante/Hacker)**: intenta minimizar dicha suma (para la defensa).

Se implementan dos algoritmos de busqueda adversarial:
1. **Minimax puro** – recorre el arbol de juego completo hasta profundidad d_max.
2. **Minimax con Poda Alfa-Beta** – optimiza el arbol eliminando ramas que no
   pueden influir en la decision optima (α ≥ β → poda).

In [12]:
import random
import copy
import math
import time
from typing import Optional, Tuple, List, Dict

import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 1. Generacion del Grafo de Red

In [13]:

SEED = 42          # reproducibilidad
NUM_NODES = 17     # entre 15 y 20 nodos
MIN_VALUE = 1      # valor minimo de informacion
MAX_VALUE = 20     # valor maximo de informacion
D_MAX = 4          # profundidad maxima de busqueda (heuristica)
TURN_LIMIT = 30    # limite de turnos para terminar el juego

def build_network(n: int = NUM_NODES, seed: int = SEED) -> nx.Graph:
    """
    Construye un grafo de red de ciberseguridad con n nodos.
    Usa Barabasi-Albert para simular topologia realista (hub-and-spoke),
    garantizando que el grafo sea conexo.
    A cada nodo se le asigna un 'info_value' entero aleatorio en [MIN_VALUE, MAX_VALUE].
    """
    rng = random.Random(seed)
    # Barabasi-Albert: m=2, grafo conexo
    G = nx.barabasi_albert_graph(n, m=2, seed=seed)
    
    for node in G.nodes():
        G.nodes[node]["info_value"] = rng.randint(MIN_VALUE, MAX_VALUE)
    return G

## 2. Estado del Juego (GameState)

In [14]:

class GameState:
    """
    Estado inmutable del juego: nodos de cada jugador, turno actual.
    """

    def __init__(
        self,
        graph: nx.Graph,
        defender_nodes: set,
        attacker_nodes: set,
        is_max_turn: bool = True,
        turn: int = 0,
    ):
        self.graph = graph
        self.defender_nodes = set(defender_nodes)
        self.attacker_nodes = set(attacker_nodes)
        self.is_max_turn = is_max_turn
        self.turn = turn

    # ------------------------------------------------------------------
    def captured_nodes(self) -> set:
        """Nodos capturados por ambos jugadores."""
        return self.defender_nodes | self.attacker_nodes

    def free_nodes(self) -> set:
        """Nodos libres."""
        return set(self.graph.nodes()) - self.captured_nodes()

    # ------------------------------------------------------------------
    def available_moves(self, for_defender: bool) -> List[int]:
        """
        Nodos libres adyacentes a los controlados por el jugador.
        """
        controlled = self.defender_nodes if for_defender else self.attacker_nodes
        candidates = set()
        for node in controlled:
            for neighbor in self.graph.neighbors(node):
                if neighbor not in self.captured_nodes():
                    candidates.add(neighbor)
        return sorted(candidates)  

    # ------------------------------------------------------------------
    def apply_move(self, node: int) -> "GameState":
        """
        Retorna nuevo GameState tras capturar node. No muta el estado actual.
        """
        new_defender = set(self.defender_nodes)
        new_attacker = set(self.attacker_nodes)
        if self.is_max_turn:
            new_defender.add(node)
        else:
            new_attacker.add(node)
        return GameState(
            self.graph,
            new_defender,
            new_attacker,
            is_max_turn=not self.is_max_turn,
            turn=self.turn + 1,
        )

    # ------------------------------------------------------------------
    def is_terminal(self) -> bool:
        """
        True si el juego termino (todos capturados, sin movimientos, o limite de turnos).
        """
        if self.turn >= TURN_LIMIT:
            return True
        if not self.free_nodes():
            return True
        
        current_is_max = self.is_max_turn
        moves = self.available_moves(for_defender=current_is_max)
        return len(moves) == 0

    # ------------------------------------------------------------------
    def terminal_score(self) -> int:
        """
        Score terminal: sum(defensor) - sum(atacante).
        """
        def_score = sum(self.graph.nodes[n]["info_value"] for n in self.defender_nodes)
        att_score = sum(self.graph.nodes[n]["info_value"] for n in self.attacker_nodes)
        return def_score - att_score

    # ------------------------------------------------------------------
    def __repr__(self):
        return (
            f"GameState(turn={self.turn}, max_turn={self.is_max_turn}, "
            f"defender={sorted(self.defender_nodes)}, "
            f"attacker={sorted(self.attacker_nodes)})"
        )

## 3. Funcion de Evaluacion Heuristica — Eval(s)

Como alcanzar nodos terminales es inviable para grafos de 17+ nodos con
branching factor alto, limitamos la busqueda a d_max = 4 capas y aplicamos
una funcion heuristica Eval(s) en los nodos hoja del arbol truncado.

**Componentes de Eval(s)**:
1. **Diferencia de valores controlados** – ventaja material directa.
2. **Diferencia de cantidad de nodos** – ventaja territorial.
3. **Potencial de expansion del defensor** – suma de valores de nodos libres
   adyacentes al defensor (oportunidades futuras de MAX).
4. **Potencial de expansion del atacante** – idem para MIN (restado).
5. **Ventaja de conectividad** – grado promedio de los nodos controlados
   (mas conexiones → mas opciones de expansion).

In [15]:

def evaluate(state: GameState) -> float:
    """
    Eval(s) heuristica: estima utilidad del estado.
    Eval = w1*material + w2*territorio + w3*frontera + w4*conectividad.
    """
    G = state.graph

    # Componente 1: ventaja material
    def_material = sum(G.nodes[n]["info_value"] for n in state.defender_nodes)
    att_material = sum(G.nodes[n]["info_value"] for n in state.attacker_nodes)
    delta_material = def_material - att_material   # positivo → ventaja defensor

    # Componente 2: ventaja territorial
    delta_territory = len(state.defender_nodes) - len(state.attacker_nodes)

    # Componente 3-4: potencial de expansion (frontera)
    def frontier_value(controlled: set) -> float:
        seen = set()
        total = 0.0
        for node in controlled:
            for nb in G.neighbors(node):
                if nb not in state.captured_nodes() and nb not in seen:
                    seen.add(nb)
                    total += G.nodes[nb]["info_value"]
        return total

    def_frontier = frontier_value(state.defender_nodes)
    att_frontier = frontier_value(state.attacker_nodes)
    delta_frontier = def_frontier - att_frontier

    # Componente 5: ventaja de conectividad
    def avg_degree(nodes: set) -> float:
        if not nodes:
            return 0.0
        return sum(G.degree(n) for n in nodes) / len(nodes)

    delta_connectivity = avg_degree(state.defender_nodes) - avg_degree(state.attacker_nodes)

    # Pesos
    w1, w2, w3, w4 = 3.0, 1.0, 1.5, 0.5

    return (
        w1 * delta_material
        + w2 * delta_territory
        + w3 * delta_frontier
        + w4 * delta_connectivity
    )

## 4. Minimax Puro

Implementacion clasica del algoritmo Minimax (Russell & Norvig, cap. 5):

```
MINIMAX-VALUE(s):
  if TERMINAL(s) or depth == 0: return EVAL(s)
  if MAX-turn:
      return max over a in ACTIONS(s): MINIMAX-VALUE(RESULT(s,a))
  else:
      return min over a in ACTIONS(s): MINIMAX-VALUE(RESULT(s,a))
```

Se añade un contador de nodos expandidos para analisis comparativo.

In [16]:

def minimax(
    state: GameState,
    depth: int,
    counter: Dict[str, int],
) -> Tuple[float, Optional[int]]:
    """
    Minimax puro. Retorna (valor, mejor_movimiento).
    """
    counter["nodes"] += 1  

    # Caso base
    if state.is_terminal() or depth == 0:
        if state.is_terminal():
            return state.terminal_score(), None
        return evaluate(state), None

    current_is_max = state.is_max_turn
    moves = state.available_moves(for_defender=current_is_max)

    
    if not moves:
        return evaluate(state), None

    best_move = None

    if current_is_max:
        # MAX maximiza
        best_val = -math.inf
        for move in moves:
            new_state = state.apply_move(move)
            val, _ = minimax(new_state, depth - 1, counter)
            if val > best_val:
                best_val = val
                best_move = move
        return best_val, best_move

    else:
        # MIN minimiza
        best_val = math.inf
        for move in moves:
            new_state = state.apply_move(move)
            val, _ = minimax(new_state, depth - 1, counter)
            if val < best_val:
                best_val = val
                best_move = move
        return best_val, best_move

## 5. Minimax con Poda Alfa-Beta

La poda alfa-beta mantiene dos valores:
- **α (alpha)**: mejor valor que MAX puede garantizar hasta ahora (inicia en -∞)
- **β (beta)**: mejor valor que MIN puede garantizar hasta ahora (inicia en +∞)

Regla de poda:
- En nodo MAX: si v ≥ β → **poda beta** (MIN nunca elegira este camino)
- En nodo MIN: si v ≤ α → **poda alfa** (MAX nunca elegira este camino)

La decision optima es IDENTICA a Minimax; solo se evitan ramas innecesarias.

```
ALPHA-BETA(s, α, β):
  if TERMINAL(s) or depth == 0: return EVAL(s)
  if MAX-turn:
      v = -∞
      for a in ACTIONS(s):
          v = max(v, ALPHA-BETA(RESULT(s,a), α, β))
          if v ≥ β: return v  ← poda beta
          α = max(α, v)
  else:
      v = +∞
      for a in ACTIONS(s):
          v = min(v, ALPHA-BETA(RESULT(s,a), α, β))
          if v ≤ α: return v  ← poda alfa
          β = min(β, v)
  return v
```

In [17]:

def alpha_beta(
    state: GameState,
    depth: int,
    alpha: float,
    beta: float,
    counter: Dict[str, int],
) -> Tuple[float, Optional[int]]:
    """
    Minimax con poda alfa-beta. Retorna (valor, mejor_movimiento).
    """
    counter["nodes"] += 1

    # Caso base
    if state.is_terminal() or depth == 0:
        if state.is_terminal():
            return state.terminal_score(), None
        return evaluate(state), None

    current_is_max = state.is_max_turn
    moves = state.available_moves(for_defender=current_is_max)

    if not moves:
        return evaluate(state), None

    best_move = None

    if current_is_max:
        # MAX maximiza
        v = -math.inf
        for move in moves:
            new_state = state.apply_move(move)
            child_val, _ = alpha_beta(new_state, depth - 1, alpha, beta, counter)
            if child_val > v:
                v = child_val
                best_move = move
            alpha = max(alpha, v)
            # Poda beta
            if v >= beta:
                break  # ← poda beta
        return v, best_move

    else:
        # MIN minimiza
        v = math.inf
        for move in moves:
            new_state = state.apply_move(move)
            child_val, _ = alpha_beta(new_state, depth - 1, alpha, beta, counter)
            if child_val < v:
                v = child_val
                best_move = move
            beta = min(beta, v)
            # Poda alfa
            if v <= alpha:
                break  # ← poda alfa
        return v, best_move

## 6. Motor del Juego

In [18]:

class GameEngine:
    """
    Motor del juego: turnos, metricas y visualizacion.
    """

    def __init__(self, graph: nx.Graph, use_alpha_beta_for_display: bool = True):
        self.graph = graph
        self.use_alpha_beta_for_display = use_alpha_beta_for_display
        self.history: List[GameState] = []
        self.metrics: List[Dict] = []  # por turno: {turn, mm_nodes, ab_nodes, move}
        self.pos = nx.spring_layout(graph, seed=SEED)  # layout fijo

    # ------------------------------------------------------------------
    def _choose_start_nodes(self) -> Tuple[int, int]:
        """
        Elige nodos iniciales: defensor toma el de mayor valor, atacante el segundo.
        """
        sorted_nodes = sorted(
            self.graph.nodes(),
            key=lambda n: (self.graph.nodes[n]["info_value"], self.graph.degree(n)),
            reverse=True,
        )
        defender_start = sorted_nodes[0]
        # Atacante: mejor nodo restante con cierta distancia al defensor
        for candidate in sorted_nodes[1:]:
            if candidate != defender_start:
                attacker_start = candidate
                break
        return defender_start, attacker_start

    # ------------------------------------------------------------------
    def initialize(self) -> GameState:
        """Crea estado inicial."""
        d_start, a_start = self._choose_start_nodes()
        state = GameState(
            self.graph,
            defender_nodes={d_start},
            attacker_nodes={a_start},
            is_max_turn=True,
            turn=0,
        )
        self.history.append(state)
        print(f"Defensor inicia en nodo {d_start} (valor={self.graph.nodes[d_start]['info_value']})")
        print(f"Atacante inicia en nodo {a_start} (valor={self.graph.nodes[a_start]['info_value']})")
        return state

    # ------------------------------------------------------------------
    def play_turn(self, state: GameState) -> Tuple[GameState, Dict]:
        """
        Ejecuta un turno: compara Minimax puro vs Alfa-Beta y aplica el movimiento.
        """
        role = "DEFENSOR (MAX)" if state.is_max_turn else "ATACANTE (MIN)"

        # --- Minimax puro ---
        mm_counter = {"nodes": 0}
        t0 = time.perf_counter()
        mm_val, mm_move = minimax(state, D_MAX, mm_counter)
        mm_time = time.perf_counter() - t0

        # --- Alfa-Beta ---
        ab_counter = {"nodes": 0}
        t0 = time.perf_counter()
        ab_val, ab_move = alpha_beta(state, D_MAX, -math.inf, math.inf, ab_counter)
        ab_time = time.perf_counter() - t0

        # Verificacion: ambos deben dar el mismo valor
        assert abs(mm_val - ab_val) < 1e-9, (
            f"Inconsistencia: Minimax={mm_val}, AlphaBeta={ab_val}"
        )

        
        chosen_move = ab_move if self.use_alpha_beta_for_display else mm_move
        chosen_move = chosen_move if chosen_move is not None else mm_move

        metric = {
            "turn": state.turn,
            "role": role,
            "move": chosen_move,
            "eval_value": ab_val,
            "mm_nodes": mm_counter["nodes"],
            "ab_nodes": ab_counter["nodes"],
            "mm_time_ms": mm_time * 1000,
            "ab_time_ms": ab_time * 1000,
            "pruning_reduction_pct": (
                100.0 * (1 - ab_counter["nodes"] / mm_counter["nodes"])
                if mm_counter["nodes"] > 0 else 0.0
            ),
        }
        self.metrics.append(metric)

        if chosen_move is not None:
            new_state = state.apply_move(chosen_move)
        else:
            
            new_state = GameState(
                state.graph,
                state.defender_nodes,
                state.attacker_nodes,
                is_max_turn=not state.is_max_turn,
                turn=state.turn + 1,
            )

        self.history.append(new_state)
        return new_state, metric

    # ------------------------------------------------------------------
    def run(self) -> GameState:
        """Ejecuta el juego completo."""
        state = self.initialize()
        print(f"\nd_max={D_MAX}, limite={TURN_LIMIT} turnos\n")

        while not state.is_terminal():
            state, metric = self.play_turn(state)
            move_str = (
                f"nodo {metric['move']} (v={self.graph.nodes[metric['move']]['info_value']})"
                if metric["move"] is not None
                else "sin movimiento"
            )
            print(
                f"T{metric['turn']:2d} {metric['role']:14s} | {move_str:25s} | "
                f"eval={metric['eval_value']:+.1f} | "
                f"MM={metric['mm_nodes']:5d} AB={metric['ab_nodes']:5d} (-{metric['pruning_reduction_pct']:.0f}%)"
            )

        self._print_final_score(state)
        return state

    # ------------------------------------------------------------------
    def _print_final_score(self, state: GameState):
        """Imprime marcador final."""
        G = self.graph
        def_score = sum(G.nodes[n]["info_value"] for n in state.defender_nodes)
        att_score = sum(G.nodes[n]["info_value"] for n in state.attacker_nodes)
        free = state.free_nodes()
        free_score = sum(G.nodes[n]["info_value"] for n in free)

        winner = "DEFENSOR" if def_score > att_score else (
            "ATACANTE" if att_score > def_score else "EMPATE"
        )
        print(f"\nResultado: Defensor={def_score} ({len(state.defender_nodes)} nodos), "
              f"Atacante={att_score} ({len(state.attacker_nodes)} nodos) -> {winner} ({def_score - att_score:+d})")

## 7. Visualizacion

In [19]:

def visualize_state(
    state: GameState,
    pos: Dict,
    title: str = "",
    ax: Optional[plt.Axes] = None,
    show: bool = False,
    save_path: Optional[str] = None,
):
    """
    Dibuja el grafo: azul=defensor, rojo=atacante, gris=libre.
    """
    G = state.graph
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 7))

    node_colors = []
    for n in G.nodes():
        if n in state.defender_nodes:
            node_colors.append("#4472C4")   # azul
        elif n in state.attacker_nodes:
            node_colors.append("#FF4444")   # rojo
        else:
            node_colors.append("#AAAAAA")   # gris libre

    labels = {n: f"{n}\n({G.nodes[n]['info_value']})" for n in G.nodes()}

    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.3, edge_color="#555555")
    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        node_color=node_colors,
        node_size=800,
        edgecolors="black",
        linewidths=1.5,
    )
    nx.draw_networkx_labels(G, pos, labels=labels, ax=ax, font_size=7, font_color="white")

    legend_handles = [
        mpatches.Patch(color="#4472C4", label="Defensor (MAX)"),
        mpatches.Patch(color="#FF4444", label="Atacante (MIN)"),
        mpatches.Patch(color="#AAAAAA", label="Libre"),
    ]
    ax.legend(handles=legend_handles, loc="upper left", fontsize=8)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.axis("off")

    if save_path:
        plt.savefig(save_path, bbox_inches="tight", dpi=120)
        print(f"  Guardado: {save_path}")
    if show:
        plt.show()

def visualize_game_snapshots(engine: GameEngine, save_prefix: str = "game_state"):
    """
    Visualiza snapshots: inicio, mitad y fin del juego.
    """
    snapshots = []
    n = len(engine.history)
    if n >= 1:
        snapshots.append((0, "Estado Inicial"))
    if n >= 3:
        mid = n // 2
        snapshots.append((mid, f"Estado Intermedio (turno {mid})"))
    if n >= 2:
        snapshots.append((n - 1, f"Estado Final (turno {n - 1})"))

    ncols = len(snapshots)
    fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 7))
    if ncols == 1:
        axes = [axes]

    for ax, (idx, title) in zip(axes, snapshots):
        visualize_state(engine.history[idx], engine.pos, title=title, ax=ax)

    plt.tight_layout()
    path = f"{save_prefix}_snapshots.png"
    plt.show()

## 8. Tabla Comparativa: Minimax vs Alpha-Beta

In [20]:

def print_comparison_table(metrics: List[Dict]):
    """
    Tabla comparativa de nodos expandidos: Minimax vs Alfa-Beta por turno.
    """
    print(
        f"{'Turno':>5} | {'Jugador':14s} | {'Nodos MM':>9} | {'Nodos AB':>9} | "
        f"{'Reduccion':>10} | {'Tiempo MM (ms)':>14} | {'Tiempo AB (ms)':>14}"
    )
    
    total_mm = total_ab = 0
    for m in metrics:
        print(
            f"{m['turn']:>5} | {m['role']:14s} | {m['mm_nodes']:>9,} | "
            f"{m['ab_nodes']:>9,} | {m['pruning_reduction_pct']:>9.1f}% | "
            f"{m['mm_time_ms']:>14.2f} | {m['ab_time_ms']:>14.2f}"
        )
        total_mm += m["mm_nodes"]
        total_ab += m["ab_nodes"]
    
    overall_reduction = (
        100.0 * (1 - total_ab / total_mm) if total_mm > 0 else 0.0
    )
    print(
        f"{'TOTAL':>5} | {'':14s} | {total_mm:>9,} | {total_ab:>9,} | "
        f"{overall_reduction:>9.1f}% | {'':>14} | {'':>14}"
    )
    
    print(f"\nResumen: Alpha-Beta expandio {overall_reduction:.1f}% menos nodos que Minimax puro.")
    print(f"En el mejor caso teorico, la poda alfa-beta alcanza O(b^(d/2)) vs O(b^d) de Minimax.")

def plot_comparison_chart(metrics: List[Dict], save_path: str = "comparison_chart.png"):
    """
    Grafico comparativo de nodos expandidos por turno.
    """
    turns = [m["turn"] for m in metrics]
    mm_nodes = [m["mm_nodes"] for m in metrics]
    ab_nodes = [m["ab_nodes"] for m in metrics]

    x = range(len(turns))
    width = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Grafico de nodos expandidos por turno
    ax1 = axes[0]
    bars1 = ax1.bar([xi - width / 2 for xi in x], mm_nodes, width,
                    label="Minimax puro", color="#4472C4", alpha=0.85)
    bars2 = ax1.bar([xi + width / 2 for xi in x], ab_nodes, width,
                    label="Alpha-Beta", color="#FF4444", alpha=0.85)
    ax1.set_xlabel("Turno")
    ax1.set_ylabel("Nodos Expandidos")
    ax1.set_title("Nodos Expandidos: Minimax vs Alpha-Beta por Turno")
    ax1.set_xticks(list(x))
    ax1.set_xticklabels([str(t) for t in turns])
    ax1.legend()
    ax1.grid(axis="y", alpha=0.3)

    # Grafico de porcentaje de reduccion
    ax2 = axes[1]
    reductions = [m["pruning_reduction_pct"] for m in metrics]
    ax2.bar(list(x), reductions, color="#70AD47", alpha=0.85)
    ax2.set_xlabel("Turno")
    ax2.set_ylabel("Reduccion (%)")
    ax2.set_title("Porcentaje de Reduccion de Nodos (Poda Alfa-Beta)")
    ax2.set_xticks(list(x))
    ax2.set_xticklabels([str(t) for t in turns])
    ax2.set_ylim(0, 100)
    ax2.axhline(y=50, color="red", linestyle="--", alpha=0.5,
                label="50% (umbral teorico minimo con orden ideal)")
    ax2.legend()
    ax2.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

## 9. Funcion Principal

In [21]:

def main():
    """Ejecuta simulacion completa: red, juego, metricas y visualizaciones."""
    print("Task 2: Defensa adversarial (Minimax / Alpha-Beta)\n")

    G = build_network(NUM_NODES, SEED)
    print(f"Red: {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas, grado promedio {sum(d for _, d in G.degree()) / G.number_of_nodes():.2f}")

    engine = GameEngine(G, use_alpha_beta_for_display=True)
    final_state = engine.run()

    print_comparison_table(engine.metrics)

    visualize_game_snapshots(engine, save_prefix="game_state")
    plot_comparison_chart(engine.metrics)

    if engine.metrics:
        avg_mm = sum(m["mm_nodes"] for m in engine.metrics) / len(engine.metrics)
        avg_ab = sum(m["ab_nodes"] for m in engine.metrics) / len(engine.metrics)
        avg_red = sum(m["pruning_reduction_pct"] for m in engine.metrics) / len(engine.metrics)
        print(f"\nResumen: d_max={D_MAX}, {len(engine.metrics)} turnos")
        print(f"  Nodos promedio - Minimax: {avg_mm:.1f}, Alpha-Beta: {avg_ab:.1f}")
        print(f"  Reduccion promedio por poda: {avg_red:.1f}%")

    return final_state, engine

main()

Task 2: Defensa adversarial (Minimax / Alpha-Beta)

Red: 17 nodos, 30 aristas, grado promedio 3.53
Defensor inicia en nodo 9 (valor=19)
Atacante inicia en nodo 7 (valor=18)

d_max=4, limite=30 turnos

T 0 DEFENSOR (MAX) | nodo 0 (v=4)              | eval=+31.2 | MM=   79 AB=   34 (-57%)
T 1 ATACANTE (MIN) | nodo 1 (v=1)              | eval=+82.8 | MM=  336 AB=  134 (-60%)
T 2 DEFENSOR (MAX) | nodo 2 (v=9)              | eval=+46.0 | MM= 1702 AB=  267 (-84%)
T 3 ATACANTE (MIN) | nodo 4 (v=8)              | eval=+84.4 | MM= 1699 AB=  559 (-67%)


T 4 DEFENSOR (MAX) | nodo 16 (v=17)            | eval=+64.0 | MM= 2931 AB= 1798 (-39%)
T 5 ATACANTE (MIN) | nodo 3 (v=8)              | eval=+81.5 | MM= 1713 AB=  191 (-89%)
T 6 DEFENSOR (MAX) | nodo 10 (v=14)            | eval=+64.0 | MM=  924 AB=  172 (-81%)
T 7 ATACANTE (MIN) | nodo 5 (v=5)              | eval=+77.0 | MM=  748 AB=  153 (-80%)
T 8 DEFENSOR (MAX) | nodo 6 (v=4)              | eval=+67.1 | MM=  430 AB=   92 (-79%)
T 9 ATACANTE (MIN) | nodo 8 (v=3)              | eval=+77.0 | MM=  191 AB=   64 (-66%)
T10 DEFENSOR (MAX) | nodo 14 (v=7)             | eval=+22.0 | MM=   70 AB=   54 (-23%)
T11 ATACANTE (MIN) | nodo 13 (v=3)             | eval=+22.0 | MM=   27 AB=   19 (-30%)
T12 DEFENSOR (MAX) | nodo 11 (v=2)             | eval=+22.0 | MM=    4 AB=    4 (-0%)
T13 ATACANTE (MIN) | nodo 15 (v=8)             | eval=+22.0 | MM=    3 AB=    3 (-0%)

Resultado: Defensor=76 (8 nodos), Atacante=54 (8 nodos) -> DEFENSOR (+22)
Turno | Jugador        |  Nodos MM |  Nodos AB |  Reducci


Resumen: d_max=4, 14 turnos
  Nodos promedio - Minimax: 775.5, Alpha-Beta: 253.1
  Reduccion promedio por poda: 53.9%


/tmp/ipykernel_17594/3374762133.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_17594/1538820678.py:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


(GameState(turn=14, max_turn=True, defender=[0, 2, 6, 9, 10, 11, 14, 16], attacker=[1, 3, 4, 5, 7, 8, 13, 15]),
 <__main__.GameEngine at 0x7f3612f292b0>)

---
# Tarea 3: Incertidumbre y Latencia (Expectiminimax y MDPs)

# Task 3 – Incertidumbre y Latencia (Expectiminimax y MDPs)

## Contexto
El juego adversarial de captura de nodos en un grafo ahora opera en un entorno
estocastico: cuando MAX o MIN intentan capturar un nodo, existe un **20% de
probabilidad de que la accion falle** y el jugador pierda su turno sin capturar nada.

## Estructura del arbol Expectiminimax
La diferencia fundamental con Minimax clasico es la insercion de **nodos de azar**
(chance nodes) entre cada capa de decision:

  MAX → Chance(p=0.8 exito, p=0.2 fallo) → MIN → Chance → MAX → ...

En cada nodo de azar, el valor esperado es:
  V(chance) = 0.8 * V(hijo_exito) + 0.2 * V(hijo_fallo)

## Reflexion MDP (escenario sin oponente)
Si no hubiera MIN, el problema se reduce a un MDP de un solo agente:

  - Estado s: frozenset de nodos capturados por MAX
  - Accion a: intentar capturar nodo vecino a
  - Transicion: con prob 0.8 → s' = s ∪ {a}; con prob 0.2 → s' = s (fallo)
  - Recompensa: exito → valor(a) − 1; fallo → −1 (costo de latencia por turno)

La Ecuacion de Bellman para V(s) es:

  V(s) = max_a [ 0.8 * (valor(a) − 1 + γ·V(s')) + 0.2 * (−1 + γ·V(s)) ]

Donde s' = s ∪ {a}.  Como V(s) aparece en ambos lados:
  V(s)·(1 − 0.2·γ) = max_a [ 0.8·valor(a) − 1 + 0.8·γ·V(s') ]

Esta es la forma canonica que usamos en value iteration mas abajo.

In [22]:
import random
import math
import statistics
from collections import defaultdict
from typing import Optional

PROB_EXITO   = 0.8   # probabilidad de que una accion tenga exito
PROB_FALLO   = 0.2   # probabilidad de que la accion falle (turno perdido)
D_MAX        = 4     # profundidad maxima del arbol de busqueda
NUM_NODOS    = 16    # nodos en el grafo del juego
NUM_JUEGOS   = 50    # partidas por matchup para analisis estadistico
LIMITE_TURNOS = 60   # limite de turnos para evitar juegos infinitos
SEMILLA_BASE = 42    # semilla para reproducibilidad

## 1. Grafo del juego (implementado sin dependencias externas)

El grafo se representa con un diccionario de adyacencia (dict[int, set])
y un diccionario de valores de nodo (dict[int, int]).

In [23]:
class Grafo:
    """
    Grafo no dirigido como lista de adyacencia.
    """

    def __init__(self, num_nodos: int):
        self.num_nodos = num_nodos
        self.nodos = list(range(num_nodos))
        self.adyacencia: dict[int, set] = {n: set() for n in self.nodos}
        self.valores: dict[int, int] = {}

    def agregar_arista(self, u: int, v: int) -> None:
        self.adyacencia[u].add(v)
        self.adyacencia[v].add(u)

    def vecinos(self, nodo: int) -> set:
        return self.adyacencia[nodo]

    def valor(self, nodo: int) -> int:
        return self.valores[nodo]

    def es_conectado(self) -> bool:
        """BFS para verificar conectividad."""
        visitados = set()
        cola = [0]
        while cola:
            nodo = cola.pop()
            if nodo in visitados:
                continue
            visitados.add(nodo)
            cola.extend(self.adyacencia[nodo] - visitados)
        return len(visitados) == self.num_nodos

def crear_grafo(num_nodos: int = NUM_NODOS, semilla: int = SEMILLA_BASE) -> Grafo:
    """
    Crea grafo aleatorio conectado (Erdos-Renyi, p=0.35). Valores de nodo en [1,10].
    """
    rng = random.Random(semilla)
    intentos = 0
    while True:
        G = Grafo(num_nodos)
        
        for u in range(num_nodos):
            for v in range(u + 1, num_nodos):
                if rng.random() < 0.35:
                    G.agregar_arista(u, v)
        if G.es_conectado():
            break
        intentos += 1
        rng = random.Random(semilla + intentos)

    
    rng2 = random.Random(semilla + 100)
    for nodo in G.nodos:
        G.valores[nodo] = rng2.randint(1, 10)
    return G

## 2. Estado del juego

In [24]:
class EstadoJuego:
    """
    Estado inmutable del juego: nodos capturados por cada jugador, turno.
    """

    __slots__ = ('grafo', 'capturados_max', 'capturados_min', 'turno', 'turno_num')

    def __init__(self, grafo: Grafo,
                 capturados_max: frozenset,
                 capturados_min: frozenset,
                 turno: str,
                 turno_num: int = 0):
        self.grafo = grafo
        self.capturados_max = capturados_max
        self.capturados_min = capturados_min
        self.turno = turno
        self.turno_num = turno_num

    def nodos_libres(self) -> frozenset:
        todos = frozenset(self.grafo.nodos)
        return todos - self.capturados_max - self.capturados_min

    def movimientos_validos(self, jugador: str) -> list:
        """
        Movimientos validos: nodos libres adyacentes a los capturados.
        """
        capturados = (self.capturados_max if jugador == 'MAX'
                      else self.capturados_min)
        libres = self.nodos_libres()
        if not capturados:
            return sorted(libres)
        vecinos_libres = set()
        for nodo in capturados:
            vecinos_libres |= (self.grafo.vecinos(nodo) & libres)
        return sorted(vecinos_libres)

    def es_terminal(self) -> bool:
        return len(self.nodos_libres()) == 0 or self.turno_num >= LIMITE_TURNOS

    def aplicar_accion(self, nodo: Optional[int], jugador: str) -> 'EstadoJuego':
        """
        Retorna nuevo estado tras la accion. None = turno perdido.
        """
        nuevo_max = self.capturados_max
        nuevo_min = self.capturados_min
        if nodo is not None:
            if jugador == 'MAX':
                nuevo_max = self.capturados_max | frozenset([nodo])
            else:
                nuevo_min = self.capturados_min | frozenset([nodo])
        siguiente = 'MIN' if jugador == 'MAX' else 'MAX'
        return EstadoJuego(
            grafo=self.grafo,
            capturados_max=nuevo_max,
            capturados_min=nuevo_min,
            turno=siguiente,
            turno_num=self.turno_num + 1,
        )

    def score_max(self) -> int:
        return sum(self.grafo.valor(n) for n in self.capturados_max)

    def score_min(self) -> int:
        return sum(self.grafo.valor(n) for n in self.capturados_min)

def estado_inicial(grafo: Grafo) -> EstadoJuego:
    return EstadoJuego(
        grafo=grafo,
        capturados_max=frozenset(),
        capturados_min=frozenset(),
        turno='MAX',
        turno_num=0,
    )

## 3. Funcion de Evaluacion Heuristica (compartida con Task 2)

Eval(s) aproxima el valor Minimax real combinando tres señales:
  1. Diferencia de puntaje acumulado (mayor peso, refleja el marcador actual).
  2. Potencial de frontera: sum valores alcanzables MAX − sum valores alcanzables MIN.
     Indica quien tiene acceso a nodos mas valiosos en el siguiente movimiento.
  3. Control territorial: nodos capturados MAX − nodos capturados MIN.

Estos tres factores capturan las dimensiones mas relevantes del estado
sin explorar el arbol hasta las hojas terminales.

In [25]:
def evaluar(estado: EstadoJuego) -> float:
    """Eval(s) heuristica para nodos no terminales."""
    # Puntaje actual
    diff_score = estado.score_max() - estado.score_min()

    # Potencial de frontera
    frontera_max = estado.movimientos_validos('MAX')
    frontera_min = estado.movimientos_validos('MIN')
    pot_max = sum(estado.grafo.valor(n) for n in frontera_max)
    pot_min = sum(estado.grafo.valor(n) for n in frontera_min)
    diff_pot = pot_max - pot_min

    # Territorio
    control = len(estado.capturados_max) - len(estado.capturados_min)

    return diff_score * 2.0 + diff_pot * 0.8 + control * 0.5

## 4. Minimax con Poda Alfa-Beta (Task 2)

Agente determinista que **asume exito garantizado** en cada accion.
No modela la probabilidad de fallo; opera como si el mundo fuera perfecto.
Se usa como baseline en el analisis comparativo.

In [26]:
class AgenteMinimaxAlfaBeta:
    """
    Minimax alfa-beta (mundo determinista, sin nodos de azar).
    """

    def __init__(self, d_max: int = D_MAX):
        self.d_max = d_max
        self.nodos_expandidos = 0

    def elegir_accion(self, estado: EstadoJuego) -> Optional[int]:
        self.nodos_expandidos = 0
        movimientos = estado.movimientos_validos('MAX')
        if not movimientos:
            return None
        mejor_val = -math.inf
        mejor_mov = movimientos[0]
        alfa, beta = -math.inf, math.inf
        for mov in movimientos:
            nuevo = estado.aplicar_accion(mov, 'MAX')
            val = self._min_valor(nuevo, 1, alfa, beta)
            if val > mejor_val:
                mejor_val = val
                mejor_mov = mov
            alfa = max(alfa, mejor_val)
        return mejor_mov

    def _max_valor(self, estado: EstadoJuego, prof: int,
                   alfa: float, beta: float) -> float:
        self.nodos_expandidos += 1
        if estado.es_terminal() or prof >= self.d_max:
            return evaluar(estado)
        movimientos = estado.movimientos_validos('MAX')
        if not movimientos:
            return evaluar(estado)
        val = -math.inf
        for mov in movimientos:
            val = max(val, self._min_valor(estado.aplicar_accion(mov, 'MAX'),
                                           prof + 1, alfa, beta))
            if val >= beta:
                return val  # poda beta
            alfa = max(alfa, val)
        return val

    def _min_valor(self, estado: EstadoJuego, prof: int,
                   alfa: float, beta: float) -> float:
        self.nodos_expandidos += 1
        if estado.es_terminal() or prof >= self.d_max:
            return evaluar(estado)
        movimientos = estado.movimientos_validos('MIN')
        if not movimientos:
            return evaluar(estado)
        val = math.inf
        for mov in movimientos:
            val = min(val, self._max_valor(estado.aplicar_accion(mov, 'MIN'),
                                           prof + 1, alfa, beta))
            if val <= alfa:
                return val  # poda alfa
            beta = min(beta, val)
        return val

## 5. Expectiminimax (Task 3)

### Relacion con la teoria
El arbol Expectiminimax extiende Minimax insertando **nodos de azar** despues
de cada capa de decision.  La estructura es:

  MAX → ChanceNode → MIN → ChanceNode → MAX → ...

**Valor de un nodo de azar** (para accion a desde estado s, jugador MAX):

  V_chance(s, a) = 0.8 · V_MIN(s ∪ {a})   <- exito: a se captura
                 + 0.2 · V_MIN(s)           <- fallo: turno perdido, sin captura

El agente MAX elige a* = argmax_a V_chance(s, a), y el agente MIN
minimiza simetricamente.

La clave es que Expectiminimax conoce la probabilidad de fallo y la
incorpora en su evaluacion.  Minimax clasico ignora el fallo y sobrestima
el valor de sus acciones.

In [27]:
class AgenteExpectiminimax:
    """
    Expectiminimax con nodos de azar (0.8 exito / 0.2 fallo).
    """

    def __init__(self, d_max: int = D_MAX):
        self.d_max = d_max
        self.nodos_expandidos = 0

    def elegir_accion(self, estado: EstadoJuego) -> Optional[int]:
        self.nodos_expandidos = 0
        movimientos = estado.movimientos_validos('MAX')
        if not movimientos:
            return None
        mejor_val = -math.inf
        mejor_mov = movimientos[0]
        for mov in movimientos:
            val = self._chance_max(estado, mov, 1)
            if val > mejor_val:
                mejor_val = val
                mejor_mov = mov
        return mejor_mov

    def _chance_max(self, estado: EstadoJuego, mov: int, prof: int) -> float:
        """
        Chance node MAX: V = 0.8*V_MIN(exito) + 0.2*V_MIN(fallo)
        """
        self.nodos_expandidos += 1
        exito = estado.aplicar_accion(mov, 'MAX')   # turno ganado
        fallo = estado.aplicar_accion(None, 'MAX')  # turno perdido
        return (PROB_EXITO * self._min_valor(exito, prof)
                + PROB_FALLO * self._min_valor(fallo, prof))

    def _chance_min(self, estado: EstadoJuego, mov: int, prof: int) -> float:
        """
        Chance node MIN: V = 0.8*V_MAX(exito) + 0.2*V_MAX(fallo)
        """
        self.nodos_expandidos += 1
        exito = estado.aplicar_accion(mov, 'MIN')
        fallo = estado.aplicar_accion(None, 'MIN')
        return (PROB_EXITO * self._max_valor(exito, prof)
                + PROB_FALLO * self._max_valor(fallo, prof))

    def _max_valor(self, estado: EstadoJuego, prof: int) -> float:
        """MAX maximiza el valor esperado."""
        self.nodos_expandidos += 1
        if estado.es_terminal() or prof >= self.d_max:
            return evaluar(estado)
        movimientos = estado.movimientos_validos('MAX')
        if not movimientos:
            return evaluar(estado)
        return max(self._chance_max(estado, mov, prof + 1) for mov in movimientos)

    def _min_valor(self, estado: EstadoJuego, prof: int) -> float:
        """MIN minimiza el valor esperado."""
        self.nodos_expandidos += 1
        if estado.es_terminal() or prof >= self.d_max:
            return evaluar(estado)
        movimientos = estado.movimientos_validos('MIN')
        if not movimientos:
            return evaluar(estado)
        return min(self._chance_min(estado, mov, prof + 1) for mov in movimientos)

## 6. Agente Aleatorio

In [28]:
class AgenteAleatorio:
    """
    Agente aleatorio: elige movimiento uniforme al azar.
    """

    def __init__(self, rng: random.Random):
        self.rng = rng

    def elegir_accion(self, estado: EstadoJuego, jugador: str) -> Optional[int]:
        movimientos = estado.movimientos_validos(jugador)
        if not movimientos:
            return None
        return self.rng.choice(movimientos)

## 7. Simulador estocastico

El **entorno** aplica la incertidumbre al ejecutar cada accion decidida.
Tanto el agente inteligente como el aleatorio son afectados por el mismo 20%
de probabilidad de fallo — esto modela la latencia de red del enunciado.

In [29]:
def simular_accion(estado: EstadoJuego, nodo: Optional[int],
                   jugador: str, rng: random.Random) -> EstadoJuego:
    """
    Ejecuta accion con 80% exito / 20% fallo.
    """
    if nodo is None:
        return estado.aplicar_accion(None, jugador)
    if rng.random() < PROB_FALLO:
        return estado.aplicar_accion(None, jugador)   # fallo
    return estado.aplicar_accion(nodo, jugador)        # exito

class ResultadoPartida:
    """Resultados de una partida."""

    def __init__(self, ganador, score_max, score_min,
                 num_turnos, nodos_exp, vals_max_intentados):
        self.ganador = ganador
        self.score_max = score_max
        self.score_min = score_min
        self.diferencia = score_max - score_min
        self.num_turnos = num_turnos
        self.nodos_exp = nodos_exp
        self.vals_max_intentados = vals_max_intentados  # para analisis agresividad

def jugar_partida(agente_max, agente_min: AgenteAleatorio,
                  grafo: Grafo, rng_env: random.Random) -> ResultadoPartida:
    """
    Juega una partida completa en entorno estocastico.
    """
    estado = estado_inicial(grafo)
    nodos_exp_total = 0
    vals_intentados = []   # valores de nodos que MAX intento capturar

    while not estado.es_terminal():
        if estado.turno == 'MAX':
            nodo = agente_max.elegir_accion(estado)
            nodos_exp_total += agente_max.nodos_expandidos
            if nodo is not None:
                vals_intentados.append(grafo.valor(nodo))
            estado = simular_accion(estado, nodo, 'MAX', rng_env)
        else:
            nodo = agente_min.elegir_accion(estado, 'MIN')
            estado = simular_accion(estado, nodo, 'MIN', rng_env)

    sc_max = estado.score_max()
    sc_min = estado.score_min()
    if sc_max > sc_min:
        ganador = 'MAX'
    elif sc_min > sc_max:
        ganador = 'MIN'
    else:
        ganador = 'EMPATE'

    return ResultadoPartida(ganador, sc_max, sc_min,
                            estado.turno_num, nodos_exp_total, vals_intentados)

## 8. Analisis comparativo: Minimax vs Expectiminimax

In [30]:
def ejecutar_analisis(num_juegos: int = NUM_JUEGOS,
                      semilla_base: int = SEMILLA_BASE) -> None:
    """
    Compara Minimax vs Expectiminimax contra agente aleatorio (50 partidas).
    """
    print(f"Analisis comparativo: {num_juegos} partidas, 20% prob. fallo\n")

    grafo = crear_grafo(NUM_NODOS, semilla=semilla_base)
    valores_grafo = [grafo.valor(n) for n in grafo.nodos]
    print(f"Grafo: {grafo.num_nodos} nodos, sum={sum(valores_grafo)}, prom={sum(valores_grafo)/len(valores_grafo):.2f}")

    res_mm: list[ResultadoPartida] = []
    res_ex: list[ResultadoPartida] = []

    for i in range(num_juegos):
        semilla_i = semilla_base + i * 31

        # Misma semilla para ambos matchups
        rng_env_mm  = random.Random(semilla_i)
        rng_min_mm  = random.Random(semilla_i + 1)
        rng_env_ex  = random.Random(semilla_i)
        rng_min_ex  = random.Random(semilla_i + 1)

        agente_mm = AgenteMinimaxAlfaBeta(d_max=D_MAX)
        agente_ex = AgenteExpectiminimax(d_max=D_MAX)

        r_mm = jugar_partida(agente_mm, AgenteAleatorio(rng_min_mm), grafo, rng_env_mm)
        r_ex = jugar_partida(agente_ex, AgenteAleatorio(rng_min_ex), grafo, rng_env_ex)

        res_mm.append(r_mm)
        res_ex.append(r_ex)

    _imprimir_resultados("MATCHUP A: Minimax alfa-beta vs Aleatorio", res_mm)
    _imprimir_resultados("MATCHUP B: Expectiminimax vs Aleatorio", res_ex)
    _analisis_agresividad(res_mm, res_ex)

def _imprimir_resultados(titulo: str, resultados: list[ResultadoPartida]) -> None:
    n = len(resultados)
    victorias = sum(1 for r in resultados if r.ganador == 'MAX')
    derrotas  = sum(1 for r in resultados if r.ganador == 'MIN')
    empates   = sum(1 for r in resultados if r.ganador == 'EMPATE')
    diffs  = [r.diferencia for r in resultados]
    turnos = [r.num_turnos for r in resultados]
    vals   = [v for r in resultados for v in r.vals_max_intentados]

    print(f"\n{titulo}")
    print(f"  Victorias: {victorias}/{n} ({100*victorias/n:.1f}%), Derrotas: {derrotas}/{n}, Empates: {empates}/{n}")
    print(f"  Dif. puntaje: prom={statistics.mean(diffs):+.2f}, mediana={statistics.median(diffs):+.2f}"
          + (f", sigma={statistics.stdev(diffs):.2f}" if n > 1 else ""))
    print(f"  Turnos por partida       – prom: {statistics.mean(turnos):.1f}")
    if vals:
        print(f"  Valor promedio de nodo intentado por MAX: {statistics.mean(vals):.2f}"
              f"  [proxy de agresividad]")

def _analisis_agresividad(res_mm: list[ResultadoPartida],
                           res_ex: list[ResultadoPartida]) -> None:
    """
    Compara agresividad: valor promedio de nodo elegido por cada agente.
    """
    vals_mm = [v for r in res_mm for v in r.vals_max_intentados]
    vals_ex = [v for r in res_ex for v in r.vals_max_intentados]
    tasa_mm = sum(1 for r in res_mm if r.ganador == 'MAX') / len(res_mm)
    tasa_ex = sum(1 for r in res_ex if r.ganador == 'MAX') / len(res_ex)

    prom_mm = statistics.mean(vals_mm) if vals_mm else 0
    prom_ex = statistics.mean(vals_ex) if vals_ex else 0

    print(f"\nAgresividad: valor promedio nodo elegido - MM={prom_mm:.2f}, EMM={prom_ex:.2f}")
    print(f"Victoria: MM={100*tasa_mm:.1f}%, EMM={100*tasa_ex:.1f}%")

    diff_vals = prom_ex - prom_mm
    if diff_vals < -0.25:
        print("Expectiminimax mas conservador: descuenta valor esperado por riesgo (0.8*v)")
    elif diff_vals > 0.25:
        print("Expectiminimax mas agresivo: busca nodos de mayor valor para compensar fallos")
    else:
        print("Agresividad similar: EMM ajusta valores (x0.8) pero mantiene la estrategia")

    delta_victoria = tasa_ex - tasa_mm
    if delta_victoria > 0.04:
        print(f"EMM gana {100*delta_victoria:.1f}% mas: modelar incertidumbre da ventaja real")
    elif delta_victoria < -0.04:
        print(f"MM gana {100*abs(delta_victoria):.1f}% mas: contra oponente debil, mundo perfecto basta")
    else:
        print("Tasas similares: la diferencia es en el razonamiento, no en victorias vs aleatorio")

## 9. Demostracion de la Ecuacion de Bellman (MDP sin oponente)

Resolvemos el MDP con value iteration para mostrar la politica optima
en un subgrafo pequeño (primeros 5 nodos).

Recordamos la ecuacion a resolver:

  V(s) = max_a [ 0.8·(valor(a) − 1 + γ·V(s ∪ {a})) + 0.2·(−1 + γ·V(s)) ]

Esta es equivalente a la forma canonica:
  V(s)·(1 − 0.2·γ) = max_a [ 0.8·valor(a) − 1 + 0.8·γ·V(s') ]

Usamos iteracion sincronica estandar hasta convergencia.

In [31]:
def demostrar_bellman(grafo: Grafo, gamma: float = 0.9,
                      num_iter: int = 300,
                      subgrafo_nodos: int = 5) -> None:
    """
    Value iteration en subgrafo. Muestra politica optima y valores convergidos.
    """
    print(f"\nValue iteration MDP (sin oponente): {subgrafo_nodos} nodos, gamma={gamma}")

    nodos = list(grafo.nodos)[:subgrafo_nodos]
    nodos_set = set(nodos)
    adj: dict[int, set] = {n: (grafo.vecinos(n) & nodos_set) for n in nodos}

    for n in nodos:
        print(f"  Nodo {n}: valor={grafo.valor(n)}, vecinos={sorted(adj[n])}")

    todos = frozenset(nodos)
    V: dict[frozenset, float] = defaultdict(float)

    for it in range(num_iter):
        V_nuevo: dict[frozenset, float] = {}
        max_delta = 0.0
        for bits in range(1 << len(nodos)):
            capturados = frozenset(
                nodos[i] for i in range(len(nodos)) if bits & (1 << i)
            )
            if capturados == todos:
                V_nuevo[capturados] = 0.0
                continue
            
            if not capturados:
                acciones = list(nodos)  # inicio: cualquier nodo
            else:
                acciones = sorted({v for n in capturados
                                   for v in adj[n] if v not in capturados})
            if not acciones:
                V_nuevo[capturados] = 0.0
                continue
            # V(s) = max_a [ 0.8*(R(a)-1 + γ*V(s')) + 0.2*(-1 + γ*V(s)) ]
            
            mejor = -math.inf
            for a in acciones:
                s_prima = capturados | frozenset([a])
                r_a = grafo.valor(a)
                val = (PROB_EXITO * (r_a - 1 + gamma * V[s_prima])
                       + PROB_FALLO * (-1 + gamma * V[capturados]))
                mejor = max(mejor, val)
            V_nuevo[capturados] = mejor
            max_delta = max(max_delta, abs(mejor - V.get(capturados, 0.0)))
        V = V_nuevo
        if max_delta < 1e-6 and it > 10:
            print(f"  Convergio en {it+1} iteraciones (Δ < 1e-6)")
            break

    
    vacio = frozenset()
    acciones_ini = list(nodos)
    print(f"\n  V(∅) = {V[vacio]:.4f}  (valor esperado desde el inicio)")
    print("\n  Politica optima desde ∅:")
    mejor_a = None
    mejor_v = -math.inf
    for a in acciones_ini:
        s_prima = frozenset([a])
        r_a = grafo.valor(a)
        val = (PROB_EXITO * (r_a - 1 + gamma * V[s_prima])
               + PROB_FALLO * (-1 + gamma * V[vacio]))
        marca = ""
        if val > mejor_v:
            mejor_v = val
            mejor_a = a
        print(f"    Capturar nodo {a} (valor={r_a:2d}): V_esperado = {val:.4f}")
    print(f"\nPolitica optima: capturar nodo {mejor_a} (valor={grafo.valor(mejor_a)})")

## 10. Funcion Principal

In [32]:
def main() -> None:
    print("Task 3: Expectiminimax y MDPs\n")

    ejecutar_analisis(num_juegos=NUM_JUEGOS, semilla_base=SEMILLA_BASE)

    grafo_demo = crear_grafo(NUM_NODOS, semilla=SEMILLA_BASE)
    demostrar_bellman(grafo_demo, gamma=0.9, num_iter=300, subgrafo_nodos=5)

    print("\nResumen:")
    print("  Minimax: MAX->MIN->... asume exito garantizado")
    print("  Expectiminimax: MAX->Chance(0.8/0.2)->MIN->Chance->... pondera valor esperado")
    print("  V_chance(s,a) = 0.8*V_MIN(s+{a}) + 0.2*V_MIN(s)")
    print("  Bellman: V(s) = max_a [0.8*(val(a)-1+g*V(s')) + 0.2*(-1+g*V(s))]")

main()

Task 3: Expectiminimax y MDPs

Analisis comparativo: 50 partidas, 20% prob. fallo

Grafo: 16 nodos, sum=95, prom=5.94



MATCHUP A: Minimax alfa-beta vs Aleatorio
  Victorias: 47/50 (94.0%), Derrotas: 3/50, Empates: 0/50
  Dif. puntaje: prom=+25.56, mediana=+28.00, sigma=17.35
  Turnos por partida       – prom: 20.5
  Valor promedio de nodo intentado por MAX: 6.85  [proxy de agresividad]

MATCHUP B: Expectiminimax vs Aleatorio
  Victorias: 45/50 (90.0%), Derrotas: 5/50, Empates: 0/50
  Dif. puntaje: prom=+26.40, mediana=+29.00, sigma=16.76
  Turnos por partida       – prom: 20.6
  Valor promedio de nodo intentado por MAX: 6.88  [proxy de agresividad]

Agresividad: valor promedio nodo elegido - MM=6.85, EMM=6.88
Victoria: MM=94.0%, EMM=90.0%
Agresividad similar: EMM ajusta valores (x0.8) pero mantiene la estrategia
Tasas similares: la diferencia es en el razonamiento, no en victorias vs aleatorio

Value iteration MDP (sin oponente): 5 nodos, gamma=0.9
  Nodo 0: valor=10, vecinos=[2, 3, 4]
  Nodo 1: valor=8, vecinos=[3]
  Nodo 2: valor=3, vecinos=[0]
  Nodo 3: valor=8, vecinos=[0, 1, 4]
  Nodo 4: valor=5,

---
## Análisis Teórico: MDP y la Ecuación de Bellman

### Formulación del MDP (sin oponente)

Cuando eliminamos al agente MIN del juego, el problema de captura de nodos
se reduce a un **Proceso de Decisión de Markov (MDP)** de un solo agente:

| Componente MDP | Definición en el problema |
|----------------|---------------------------|
| **Estado** s | frozenset de nodos capturados por MAX |
| **Acción** a | Intentar capturar un nodo vecino libre |
| **Transición** T(s, a, s') | P(s'=s∪{a} \| s,a) = 0.8;  P(s'=s \| s,a) = 0.2 |
| **Recompensa** R(s, a) | Éxito: valor(a) − 1;  Fallo: −1 |
| **Factor de descuento** γ | 0.9 (ejemplo) |

### Ecuación de Bellman

La **ecuación de Bellman** para el valor óptimo V*(s) expresa que el valor
de un estado es el máximo sobre las acciones del valor esperado de aplicar
esa acción y seguir la política óptima:

$$V(s) = \max_a \left[ 0.8 \cdot \bigl(\text{valor}(a) - 1 + \gamma \cdot V(s')\bigr)
         + 0.2 \cdot \bigl(-1 + \gamma \cdot V(s)\bigr) \right]$$

donde $s' = s \cup \{a\}$.

Como $V(s)$ aparece en ambos lados, reordenando:

$$V(s) \cdot (1 - 0.2\gamma) = \max_a \left[ 0.8 \cdot \text{valor}(a) - 1
         + 0.8\gamma \cdot V(s') \right]$$

Esta es la **forma canónica** usada en la iteración de valor implementada en
`demostrar_bellman()`.

### Relación con Expectiminimax

El árbol Expectiminimax es la extensión natural del MDP al entorno de dos
agentes.  La diferencia clave:

- **MDP**: MAX elige, la naturaleza introduce incertidumbre (nodo de azar).
- **Expectiminimax**: MAX elige → nodo de azar → MIN elige → nodo de azar → ...

El **valor de un nodo de azar** (tras movimiento de MAX eligiendo acción $a$
desde estado $s$) es:

$$V_{\text{chance}}(s, a) = 0.8 \cdot V_{\text{MIN}}(s \cup \{a\})
                          + 0.2 \cdot V_{\text{MIN}}(s)$$

MAX entonces elige $a^* = \arg\max_a V_{\text{chance}}(s, a)$.

### Diferencia entre Minimax clásico y Expectiminimax en entornos estocásticos

| Aspecto | Minimax (Task 2) | Expectiminimax (Task 3) |
|---------|-----------------|------------------------|
| Modelo del mundo | Determinista (asume éxito 100%) | Estocástico (modela 20% de fallo) |
| Árbol de búsqueda | MAX → MIN → MAX → ... | MAX → Chance → MIN → Chance → ... |
| Valor de acción $a$ | valor(a) (sin descuento por riesgo) | $0.8 \cdot$ valor(a) |
| Agresividad | Puede sobrestimar nodos de alto valor | Más conservador ante el riesgo |

La principal ventaja de Expectiminimax es que **razona correctamente** sobre
las consecuencias reales de sus decisiones, mientras que Minimax podría
seleccionar estrategias que dependen de éxito garantizado y que en la
práctica rinden peor ante la incertidumbre.